# 09 - Phân tích chi tiết các ca dự đoán sai trên tập Validation (S4-06)

**Bài tập lớn môn Học máy:** Phân loại 4 nhóm ảnh X-quang ngực (`COVID`, `Lung_Opacity`, `Normal`, `Viral Pneumonia`).  
**Hạng mục:** `S4-06` — Phân tích chi tiết các ca dự đoán sai (False Positives, False Negatives) của mô hình tối ưu đã đóng băng ở S4-04 (`rq2_ghep_svm_s42`: `SVC(kernel='rbf', C=10.0, gamma='scale', class_weight='balanced')` + `StandardScaler` trên đặc trưng `ghep` 152 chiều).  
**Người phụ trách (Code Owner):** Lê Minh (Cặp 3).

## Mục tiêu phân tích (tuân thủ Cổng H2 & H6 — tuyệt đối không chạm tập `test`):
1. **Phân rã toàn bộ 349/3.174 ca dự đoán sai trên tập `val`** theo 12 cặp nhầm lẫn có hướng (`true_label -> pred_label`).
2. **Định danh các ca khó đồng thuận (Consensus Hard Errors):** Đối chiếu dự đoán của `svm` với 2 họ mô hình mạnh tiếp theo (`gb` và `rf` lấy đa số qua 3 seed `42, 43, 44`) để tách biệt lỗi đặc thù của biên quyết định RBF-SVM với lỗi nội tại của đặc trưng/nhãn dữ liệu.
3. **Lập danh sách các ca nhầm lẫn lâm sàng điển hình** giữa `COVID`, `Lung_Opacity`, `Viral Pneumonia` và `Normal` (gồm cả nhóm sai tự tin cao `proba_pred >= 0,80` và nhóm cận biên `margin < 0,15`).
4. **Phân tích ảnh hưởng của độ sáng trung bình (`mean_brightness`), chế độ kênh màu (`color_mode`) và cụm trùng lặp `pHash` (`group_id`)** lên tỷ lệ dự đoán sai.


In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "results" / "threshold_v3.md").exists() and (REPO_ROOT.parent / "results" / "threshold_v3.md").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.metrics import DEFAULT_CLASS_NAMES, compute_metrics

RESULTS_DIR = REPO_ROOT / "results"
ERROR_DIR = RESULTS_DIR / "error_analysis"
ERROR_DIR.mkdir(parents=True, exist_ok=True)

PROBA_COLS = ["proba_COVID", "proba_Lung_Opacity", "proba_Normal", "proba_Viral_Pneumonia"]
LABEL_MAP = {idx: name for idx, name in enumerate(DEFAULT_CLASS_NAMES)}

# Đọc dự đoán của mô hình đóng băng rq2_ghep_svm_s42 trên tập val (3.174 ảnh)
svm_df = pd.read_csv(RESULTS_DIR / "rq2_ghep_svm_s42" / "preds_val.csv").sort_values("image_id").reset_index(drop=True)

# Đọc dự đoán 3 seed của gb và rf để xác định các ca khó đồng thuận (consensus hard errors)
gb_preds = np.stack(
    [
        pd.read_csv(RESULTS_DIR / f"rq2_ghep_gb_s{s}" / "preds_val.csv")
        .sort_values("image_id")["y_pred"]
        .to_numpy(dtype=np.int64)
        for s in (42, 43, 44)
    ],
    axis=0,
)
rf_preds = np.stack(
    [
        pd.read_csv(RESULTS_DIR / f"rq2_ghep_rf_s{s}" / "preds_val.csv")
        .sort_values("image_id")["y_pred"]
        .to_numpy(dtype=np.int64)
        for s in (42, 43, 44)
    ],
    axis=0,
)

# Đọc metadata từ splits/split_v1.csv (chỉ lấy val, tuyệt đối không đọc test) và splits/manifest.csv
split_df = pd.read_csv(REPO_ROOT / "splits" / "split_v1.csv")
val_split_df = split_df[split_df["split"] == "val"][["image_id", "group_id", "phash"]].copy()
assert len(val_split_df) == 3174, "Chỉ phân tích trên đúng 3.174 ảnh validation!"

manifest_df = pd.read_csv(REPO_ROOT / "splits" / "manifest.csv")[
    ["image_id", "mean_brightness", "color_mode", "rel_image_path"]
]

full_val_df = (
    svm_df.merge(val_split_df, on="image_id", how="inner")
    .merge(manifest_df, on="image_id", how="inner")
    .sort_values("image_id")
    .reset_index(drop=True)
)
assert len(full_val_df) == 3174

y_true_arr = full_val_df["y_true"].to_numpy(dtype=np.int64)
y_pred_arr = full_val_df["y_pred"].to_numpy(dtype=np.int64)
probas = full_val_df[PROBA_COLS].to_numpy(dtype=np.float64)

full_val_df["true_label"] = [LABEL_MAP[int(y)] for y in y_true_arr]
full_val_df["pred_label"] = [LABEL_MAP[int(y)] for y in y_pred_arr]
full_val_df["proba_true"] = probas[np.arange(len(full_val_df)), y_true_arr]
full_val_df["proba_pred"] = probas[np.arange(len(full_val_df)), y_pred_arr]
full_val_df["margin"] = full_val_df["proba_pred"] - full_val_df["proba_true"]

# Đa số phiếu dự đoán sai trên 3 seed của gb và rf (>= 2/3 seed đoán sai)
full_val_df["wrong_in_gb"] = np.sum(gb_preds != y_true_arr[None, :], axis=0) >= 2
full_val_df["wrong_in_rf"] = np.sum(rf_preds != y_true_arr[None, :], axis=0) >= 2
full_val_df["is_error"] = full_val_df["y_true"] != full_val_df["y_pred"]
full_val_df["consensus_hard_error"] = (
    full_val_df["is_error"] & full_val_df["wrong_in_gb"] & full_val_df["wrong_in_rf"]
)

def classify_error_tier(row: pd.Series) -> str:
    if not bool(row["is_error"]):
        return "Correct"
    p = float(row["proba_pred"])
    if p >= 0.80:
        return "High-confidence error (p >= 0.80)"
    if p >= 0.55:
        return "Moderate error (0.55 <= p < 0.80)"
    return "Borderline error (p < 0.55)"

full_val_df["error_type"] = full_val_df.apply(classify_error_tier, axis=1)

mis_df = full_val_df[full_val_df["is_error"]].copy().reset_index(drop=True)
mis_df.to_csv(ERROR_DIR / "val_misclassified_cases.csv", index=False)

print(f"Tổng số ảnh validation: {len(full_val_df)}")
print(f"Số ca dự đoán đúng (SVM): {len(full_val_df) - len(mis_df)} ({(1 - len(mis_df)/len(full_val_df))*100:.2f}%)")
print(f"Số ca dự đoán sai (SVM):  {len(mis_df)} ({len(mis_df)/len(full_val_df)*100:.2f}%)")
print(f"Số ca sai đồng thuận cả 3 mô hình mạnh nhất (SVM + GB + RF): {int(mis_df['consensus_hard_error'].sum())}/{len(mis_df)} ({mis_df['consensus_hard_error'].mean()*100:.1f}%)")
print("\nPhân bố mức độ tự tin của 349 ca dự đoán sai:")
print(mis_df["error_type"].value_counts().to_string())


Tổng số ảnh validation: 3174
Số ca dự đoán đúng (SVM): 2825 (89.00%)
Số ca dự đoán sai (SVM):  349 (11.00%)
Số ca sai đồng thuận cả 3 mô hình mạnh nhất (SVM + GB + RF): 240/349 (68.8%)

Phân bố mức độ tự tin của 349 ca dự đoán sai:
error_type
Moderate error (0.55 <= p < 0.80)    143
High-confidence error (p >= 0.80)    138
Borderline error (p < 0.55)           68


## 1. Bảng tổng hợp 12 cặp nhầm lẫn có hướng (`true_label -> pred_label`)

Phân tích toàn bộ 12 cặp ngoài đường chéo chính của ma trận nhầm lẫn $4 	imes 4$ trên tập `val`, tính tỷ trọng phần trăm trên tổng số lỗi (`349` ca), tỷ lệ lỗi trên tổng số mẫu thực tế của lớp gốc (`class_support`), xác suất dự đoán trung bình (`mean_proba_pred`), tỷ lệ lỗi đồng thuận trên cả 3 mô hình (`consensus_rate`), và các mã ảnh (`image_id`) đại diện có độ tự tin sai cao nhất.


In [2]:
pair_rows = []
class_supports = full_val_df.groupby("true_label").size().to_dict()

for t_idx, t_name in enumerate(DEFAULT_CLASS_NAMES):
    for p_idx, p_name in enumerate(DEFAULT_CLASS_NAMES):
        if t_idx == p_idx:
            continue
        sub = mis_df[(mis_df["y_true"] == t_idx) & (mis_df["y_pred"] == p_idx)].sort_values(
            "proba_pred", ascending=False
        )
        cnt = len(sub)
        supp = int(class_supports[t_name])
        top_examples = ", ".join(sub["image_id"].head(3).tolist()) if cnt > 0 else "None"
        pair_rows.append({
            "true_label": t_name,
            "pred_label": p_name,
            "count": cnt,
            "pct_of_total_errors": round(cnt / len(mis_df) * 100.0, 2),
            "pct_of_true_class": round(cnt / supp * 100.0, 2),
            "mean_proba_pred": round(float(sub["proba_pred"].mean()), 4) if cnt > 0 else 0.0,
            "mean_proba_true": round(float(sub["proba_true"].mean()), 4) if cnt > 0 else 0.0,
            "high_conf_count": int((sub["proba_pred"] >= 0.80).sum()) if cnt > 0 else 0,
            "consensus_3models_count": int(sub["consensus_hard_error"].sum()) if cnt > 0 else 0,
            "mean_brightness": round(float(sub["mean_brightness"].mean()), 2) if cnt > 0 else 0.0,
            "top_representative_ids": top_examples,
        })

pairs_df = pd.DataFrame(pair_rows).sort_values("count", ascending=False).reset_index(drop=True)
pairs_df.to_csv(ERROR_DIR / "confusion_pairs_summary.csv", index=False)

print("=== BẢNG PHÂN RÃ 12 CẶP NHẦM LẪN NGOÀI ĐƯỜNG CHÉO CHÍNH (SẮP XẾP THEO SỐ LƯỢNG GIẢM DẦN) ===")
display_pair_cols = [
    "true_label",
    "pred_label",
    "count",
    "pct_of_total_errors",
    "pct_of_true_class",
    "mean_proba_pred",
    "high_conf_count",
    "consensus_3models_count",
    "top_representative_ids",
]
print(pairs_df[display_pair_cols].to_string(index=False))


=== BẢNG PHÂN RÃ 12 CẶP NHẦM LẪN NGOÀI ĐƯỜNG CHÉO CHÍNH (SẮP XẾP THEO SỐ LƯỢNG GIẢM DẦN) ===
     true_label      pred_label  count  pct_of_total_errors  pct_of_true_class  mean_proba_pred  high_conf_count  consensus_3models_count                                                     top_representative_ids
         Normal    Lung_Opacity    144                41.26               9.41           0.6933               46                       98                          Normal-9540.png, Normal-4930.png, Normal-8652.png
   Lung_Opacity          Normal    124                35.53              13.73           0.8127               71                      101        Lung_Opacity-3644.png, Lung_Opacity-2478.png, Lung_Opacity-5495.png
         Normal           COVID     18                 5.16               1.18           0.5989                5                        5                          Normal-3652.png, Normal-7599.png, Normal-1709.png
         Normal Viral Pneumonia     16                 

## 2. Danh sách các ca nhầm lẫn lâm sàng điển hình giữa `COVID`, `Lung_Opacity`, `Viral Pneumonia` và `Normal`

Theo yêu cầu nghiệm thu `S4-06`, nhóm lập danh sách các ca nhầm lẫn điển hình chia theo 6 nhóm cặp bệnh lý đối xứng (`COVID <-> Lung_Opacity`, `COVID <-> Viral Pneumonia`, `Lung_Opacity <-> Viral Pneumonia`, `COVID <-> Normal`, `Viral Pneumonia <-> Normal`, và `Lung_Opacity <-> Normal`), bao gồm cả các ca **sai với độ tự tin rất cao** (`High-confidence`, nghi ngờ nhiễu nhãn hoặc bất thường hình thái mạnh) và các ca **nhầm lẫn vùng biên** (`Borderline`, đặc trưng chồng lấp giữa hai bệnh lý).


In [3]:
def assign_clinical_group(row: pd.Series) -> str:
    pair = {str(row["true_label"]), str(row["pred_label"])}
    if pair == {"COVID", "Lung_Opacity"}:
        return "COVID <-> Lung_Opacity"
    if pair == {"COVID", "Viral Pneumonia"}:
        return "COVID <-> Viral Pneumonia"
    if pair == {"Lung_Opacity", "Viral Pneumonia"}:
        return "Lung_Opacity <-> Viral Pneumonia"
    if pair == {"COVID", "Normal"}:
        return "COVID <-> Normal"
    if pair == {"Viral Pneumonia", "Normal"}:
        return "Viral Pneumonia <-> Normal"
    return "Lung_Opacity <-> Normal"

mis_df["clinical_group"] = mis_df.apply(assign_clinical_group, axis=1)

typical_rows = []
for grp in [
    "COVID <-> Lung_Opacity",
    "COVID <-> Viral Pneumonia",
    "Lung_Opacity <-> Viral Pneumonia",
    "COVID <-> Normal",
    "Viral Pneumonia <-> Normal",
    "Lung_Opacity <-> Normal",
]:
    sub = mis_df[mis_df["clinical_group"] == grp].copy()
    if sub.empty:
        continue
    # Lấy tối đa 4 ca tự tin sai cao nhất và 2 ca cận biên nhất cho mỗi hướng trong nhóm
    for (t_lbl, p_lbl), dir_df in sub.groupby(["true_label", "pred_label"]):
        dir_sorted = dir_df.sort_values("proba_pred", ascending=False)
        top_hi = dir_sorted.head(3)
        bot_lo = dir_sorted.tail(2) if len(dir_sorted) > 3 else dir_sorted.iloc[0:0]
        picked = pd.concat([top_hi, bot_lo], axis=0).drop_duplicates(subset=["image_id"])
        for _, r in picked.iterrows():
            typical_rows.append({
                "clinical_group": grp,
                "direction": f"{t_lbl} -> {p_lbl}",
                "image_id": r["image_id"],
                "true_label": t_lbl,
                "pred_label": p_lbl,
                "proba_pred": round(float(r["proba_pred"]), 4),
                "proba_true": round(float(r["proba_true"]), 4),
                "margin": round(float(r["margin"]), 4),
                "error_type": r["error_type"],
                "consensus_hard_error": bool(r["consensus_hard_error"]),
                "mean_brightness": round(float(r["mean_brightness"]), 2),
                "group_id": r["group_id"],
            })

typical_df = pd.DataFrame(typical_rows)
typical_df.to_csv(ERROR_DIR / "typical_clinical_confusions.csv", index=False)

print(f"Đã trích xuất {len(typical_df)} ca nhầm lẫn điển hình vào results/error_analysis/typical_clinical_confusions.csv")
print("\n=== CÁC CA NHẦM LẪN ĐIỂN HÌNH GIỮA COVID, LUNG OPACITY VÀ VIRAL PNEUMONIA ===")
focus_df = typical_df[
    typical_df["clinical_group"].isin([
        "COVID <-> Lung_Opacity",
        "COVID <-> Viral Pneumonia",
        "Lung_Opacity <-> Viral Pneumonia",
    ])
]
print(focus_df.to_string(index=False))


Đã trích xuất 47 ca nhầm lẫn điển hình vào results/error_analysis/typical_clinical_confusions.csv

=== CÁC CA NHẦM LẪN ĐIỂN HÌNH GIỮA COVID, LUNG OPACITY VÀ VIRAL PNEUMONIA ===
                  clinical_group                       direction                image_id      true_label      pred_label  proba_pred  proba_true  margin                        error_type  consensus_hard_error  mean_brightness group_id
          COVID <-> Lung_Opacity           COVID -> Lung_Opacity          COVID-1183.png           COVID    Lung_Opacity      0.9870      0.0099  0.9771 High-confidence error (p >= 0.80)                  True           122.50   G00204
          COVID <-> Lung_Opacity           COVID -> Lung_Opacity           COVID-329.png           COVID    Lung_Opacity      0.9029      0.0704  0.8325 High-confidence error (p >= 0.80)                  True           155.53   G02465
          COVID <-> Lung_Opacity           COVID -> Lung_Opacity          COVID-1084.png           COVID    Lung_Opaci

## 3. Phân tích đặc trưng độ sáng (`mean_brightness`) giữa nhóm dự đoán Đúng và Sai & Xuất báo cáo tổng hợp `S4-06`

Đối chiếu độ sáng trung bình của các ca dự đoán đúng so với các ca bị nhầm lẫn theo từng lớp để kiểm tra giả thuyết: liệu các ảnh `COVID` bị đoán nhầm sang `Normal`/`Lung_Opacity` có độ sáng thấp hơn mức đặc trưng của lớp `COVID` (`~151,3`), hoặc các ảnh `Normal` bị đoán nhầm sang `COVID` có độ sáng cao bất thường hay không.


In [4]:
brightness_stats = (
    full_val_df.groupby(["true_label", "is_error"], as_index=False)
    .agg(
        n_samples=("image_id", "count"),
        brightness_mean=("mean_brightness", "mean"),
        brightness_std=("mean_brightness", "std"),
        proba_pred_mean=("proba_pred", "mean"),
    )
)
print("=== SO SÁNH ĐỘ SÁNG TRUNG BÌNH GIỮA NHÓM ĐOÁN ĐÚNG (is_error=False) VÀ ĐOÁN SAI (is_error=True) ===")
print(brightness_stats.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

# Tạo báo cáo tổng hợp Markdown tại results/error_analysis/error_analysis_summary.md
summary_md_lines = [
    "# Báo cáo phân tích chi tiết các ca dự đoán sai trên tập Validation (S4-06)",
    "",
    "Ngày ghi: 07/11/2026",
    "Người thực hiện: Lê Minh (Cặp 3) — Kiểm duyệt: Nguyễn Minh Thuận & Trần Nguyễn Đức",
    "Mô hình phân tích: `rq2_ghep_svm_s42` (`SVC(kernel='rbf', C=10.0, gamma='scale', class_weight='balanced')` + `StandardScaler` trên đặc trưng `ghep` 152 chiều, đã đóng băng tại `S4-04`).",
    "Tập dữ liệu: `3.174` ảnh Validation của `splits/split_v1.csv` (`allow_test = False`).",
    "",
    "## 1. Tổng quan phân bố lỗi",
    f"- **Tổng số ca dự đoán sai:** `{len(mis_df)} / {len(full_val_df)}` ảnh (`{len(mis_df)/len(full_val_df)*100:.2f}%`; Accuracy = `89,00%`, Macro-F1 = `0,9024`).",
    f"- **Số ca khó đồng thuận (Consensus Hard Errors — cả 3 mô hình `svm`, `gb`, `rf` đều đoán sai):** `{int(mis_df['consensus_hard_error'].sum())} / {len(mis_df)}` ca (`{mis_df['consensus_hard_error'].mean()*100:.1f}%`). Điều này chứng tỏ hơn một nửa số ca sai không phải do hạn chế riêng của bộ phân loại SVM mà xuất phát từ sự chồng lấp đặc trưng mức xám/kết cấu hoặc nhiễu nhãn vùng biên giữa các lớp.",
    f"- **Phân tầng theo độ tự tin dự đoán (`proba_pred`):**",
    f"  - Sai tự tin cao (`p >= 0,80`): `{int((mis_df['proba_pred'] >= 0.80).sum())}` ca (`{(mis_df['proba_pred'] >= 0.80).mean()*100:.1f}%`).",
    f"  - Sai mức trung bình (`0,55 <= p < 0,80`): `{int(((mis_df['proba_pred'] >= 0.55) & (mis_df['proba_pred'] < 0.80)).sum())}` ca.",
    f"  - Sai vùng ranh giới (`p < 0,55`): `{int((mis_df['proba_pred'] < 0.55).sum())}` ca.",
    "",
    "## 2. Phân rã 12 cặp nhầm lẫn ngoài đường chéo chính",
    "",
    "| Nhãn thực (`true_label`) | Nhãn dự đoán (`pred_label`) | Số ca (`count`) | % Tổng lỗi | % Lớp thực | Xác suất sai TB | Số ca `p >= 0,80` | Đồng thuận 3 mô hình | Các ca điển hình (`image_id`) |",
    "| :--- | :--- | :---: | :---: | :---: | :---: | :---: | :---: | :--- |",
]

for _, r in pairs_df.iterrows():
    summary_md_lines.append(
        f"| `{r['true_label']}` | `{r['pred_label']}` | **{int(r['count'])}** | {r['pct_of_total_errors']:.2f}% | "
        f"{r['pct_of_true_class']:.2f}% | {r['mean_proba_pred']:.4f} | {int(r['high_conf_count'])} | "
        f"{int(r['consensus_3models_count'])} | `{r['top_representative_ids']}` |"
    )

summary_md_lines.extend([
    "",
    "## 3. Nguyên nhân gây nhầm lẫn theo từng cụm bệnh lý lâm sàng",
    "1. **Cụm nhầm lẫn áp đảo `Normal <-> Lung_Opacity` (`268 / 349` ca = `76,79%` tổng số lỗi):**",
    "   - Gồm `144` ca `Normal -> Lung_Opacity` và `124` ca `Lung_Opacity -> Normal` (khớp tuyệt đối với ma trận con R2).",
    "   - **Nguyên nhân:** Cả hai nhóm này đều có nguồn gốc chủ yếu từ cuộc thi `RSNA Pneumonia Detection Challenge` (`6.012` ảnh `Lung_Opacity` và `8.851/10.192` ảnh `Normal`), dùng chung thiết bị chụp và tiền xử lý nên histogram độ xám toàn cục và kết cấu nền rất giống nhau. Các ca mờ phổi nhẹ hoặc khu trú chỉ làm thay đổi cục bộ một phần nhỏ diện tích phổi, bị trung bình hoá khi tính đặc trưng toàn ảnh `299x299`.",
    "2. **Cụm nhầm lẫn `COVID <-> Lung_Opacity` (`20` ca: `12` `Lung_Opacity -> COVID` và `8` `COVID -> Lung_Opacity`):**",
    "   - Các ca điển hình: `Lung_Opacity-3062.png`, `Lung_Opacity-5040.png`, `COVID-1183.png`, `COVID-329.png`.",
    "   - **Nguyên nhân:** Tổn thương kính mờ (ground-glass opacity) lan tỏa của COVID-19 và mờ phổi không do COVID có cấu trúc mức xám và độ tương phản cục bộ (GLCM/LBP) chồng lấp; các ảnh `Lung_Opacity` có độ sáng trung bình cao dễ bị kéo về phía phân bố của `COVID`.",
    "3. **Cụm nhầm lẫn `COVID <-> Normal` (`31` ca: `18` `Normal -> COVID` và `13` `COVID -> Normal`):**",
    "   - Các ca điển hình: `Normal-3652.png`, `Normal-7599.png`, `COVID-3419.png`, `COVID-32.png`.",
    "   - **Nguyên nhân:** Phù hợp với nhận xét của chính tác giả bài báo gốc P1 (Chowdhury et al., 2020): các ca COVID-19 giai đoạn sớm có rất ít dấu hiệu mờ trên X-quang, trong khi một số ảnh `Normal` có độ sáng cao hoặc viền chữ/khung chụp đặc trưng nguồn gây nhiễu đặc trưng toàn ảnh.",
    "4. **Cụm nhầm lẫn liên quan `Viral Pneumonia` (`30` ca: `16` `Normal -> Viral Pneumonia`, `7` `Viral Pneumonia -> Normal`, `6` `COVID <-> Viral Pneumonia`, `1` `Viral Pneumonia -> Lung_Opacity`):**",
    "   - Lớp `Viral Pneumonia` (`1.345` ảnh) và `1.341` ảnh `Normal` cùng xuất phát từ bộ Kaggle Nhi khoa (Kermany/Mooney), nên sự nhầm lẫn của `Viral Pneumonia` diễn ra chủ yếu với phần `Normal` cùng nguồn (`23/30` ca), trong khi hầu như tách biệt với `Lung_Opacity` của nguồn RSNA (chỉ `1` ca duy nhất `Viral Pneumonia-622.png` và `0` ca `Lung_Opacity -> Viral Pneumonia`). Đây là bằng chứng quan trọng gợi ý hiện tượng phụ thuộc nguồn dữ liệu (shortcut learning) sẽ được kiểm định trực tiếp bằng bộ ảnh che phổi ở **RQ3 (`S4-07`, `S5-01`, `S5-02`)**.",
])

(ERROR_DIR / "error_analysis_summary.md").write_text("\n".join(summary_md_lines) + "\n", encoding="utf-8")
print("\nĐã xuất báo cáo tổng hợp S4-06 tại results/error_analysis/error_analysis_summary.md")


=== SO SÁNH ĐỘ SÁNG TRUNG BÌNH GIỮA NHÓM ĐOÁN ĐÚNG (is_error=False) VÀ ĐOÁN SAI (is_error=True) ===
     true_label  is_error  n_samples  brightness_mean  brightness_std  proba_pred_mean
          COVID     False        514           142.03           24.31             0.96
          COVID      True         23           134.93           19.10             0.72
   Lung_Opacity     False        767           124.60           22.81             0.85
   Lung_Opacity      True        136           129.77           26.69             0.80
         Normal     False       1353           129.99           21.92             0.91
         Normal      True        178           129.52           23.55             0.67
Viral Pneumonia     False        191           123.46           18.90             0.91
Viral Pneumonia      True         12           122.20           24.25             0.70

Đã xuất báo cáo tổng hợp S4-06 tại results/error_analysis/error_analysis_summary.md
